# 01 — Data exploration

> This system predicts glucose values for research and analysis. It is not a medical device and must not be used to determine insulin doses or make treatment decisions.

Reads the clean tables built by `python -m src.ingest`. Run from the project root (`uv run --extra notebooks jupyter lab`).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
import pandas as pd
from src import config, db, validation as v
settings = config.settings()
TZ = settings['project']['timezone']
duck = db.duck(read_only=True)
q = lambda sql: duck.execute(sql).df()

## Tables

In [ ]:
q("""SELECT table_name, estimated_size AS rows FROM duckdb_tables() ORDER BY table_name""")

## Data-quality issues

In [ ]:
q("SELECT source_table, issue_type, severity, count(*) n FROM dq_issues GROUP BY ALL ORDER BY n DESC")

## CGM summary (modelling window)

In [ ]:
w = settings['modelling_window']
start = pd.Timestamp(w['start'], tz=TZ).tz_convert('UTC')
end = (pd.Timestamp(w['end'], tz=TZ) + pd.Timedelta(days=1)).tz_convert('UTC')
merged = q('SELECT * FROM cgm_merged')
merged['measured_at'] = pd.to_datetime(merged['measured_at'], utc=True)
pd.Series(v.cgm_metrics(merged, settings, start, end))

## Backfill correction

Backfilled Tandem readings arrive late; how late?

In [ ]:
lag = q("""SELECT epoch(available_at - measured_at)/60 AS minutes_late FROM cgm_readings WHERE time_corrected""")
lag['minutes_late'].describe(percentiles=[.5, .9, .99])

## Bolus types

In [ ]:
q("""SELECT delivery_type, delivery_shape, count(*) n, round(avg(units_delivered), 2) avg_u, round(sum(units_delivered)) total_u,
  count(*) FILTER (WHERE carbs_g > 0) with_carbs FROM boluses GROUP BY ALL ORDER BY n DESC""")

## One week, aligned

In [ ]:
from src import plots
from IPython.display import Image
segments = q('SELECT * FROM basal_segments')
boluses = q('SELECT * FROM boluses')
for df, cols in ((segments, ['start_at', 'end_at']), (boluses, ['requested_at', 'completed_at'])):
    for c in cols:
        df[c] = pd.to_datetime(df[c], utc=True)
day = pd.Timestamp('2026-02-09', tz=TZ).tz_convert('UTC')
p = plots.aligned_day(merged, segments, boluses, day, day + pd.Timedelta(days=1), TZ, settings['glucose_thresholds_mmol'], config.path('reports/figures') / 'notebook_aligned.png')
Image(str(p))